In [1]:
import os
import gc
import joblib
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb

from datetime import datetime, timedelta
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ==========================
# Config & constants
# ==========================

CLEAN_PATH = r"C:\grad2_out\taxi_with_weather_FULL_dataset"
MODEL_PATH = r"C:\grad2_out\eta_model_artifact.joblib"
FI_PATH = r"C:\grad2_out\lgbm_eta_feature_importance_v2.csv"

TRAIN_END = datetime(2024, 11, 1)
VALID_END = datetime(2024, 12, 1)

MAX_TRAIN_ROWS = 3_000_000
MAX_VALID_ROWS = 500_000
MAX_TEST_ROWS = 500_000

SEED = 42
RNG = np.random.default_rng(SEED)

TARGET = "duration_sec"
MIN_DURATION = 60.0
MAX_DURATION = 7200.0

# distance buckets in km
DIST_BUCKET_EDGES = [0.0, 2.0, 5.0, 10.0, 20.0, 50.0]

MODEL_VERSION = "eta_v1.0_quantile"
FEATURE_VERSION = "v1_base_od_stats"

CATEGORICAL_MISSING_TOKEN = "__MISSING__"
CATEGORICAL_UNK_TOKEN = "__UNK__"

MIN_SUPPORT_OD_15MIN = 20
MIN_SUPPORT_OD_HOUR = 20
MIN_SUPPORT_DIST_BUCKET = 30

# حدود تنظيف البيانات ومعالجة الشواذ (قبل أي تدريب)
DURATION_CLIP = (60.0, 7200.0)
DISTANCE_KM_CLIP = (0.1, 50.0)
TEMP_C_CLIP = (-20.0, 50.0)
RAIN_MM_CLIP = (0.0, 150.0)
SPEED_SEC_PER_KM_CLIP = (30.0, 600.0)
CLEAN_REPORT = True

# تدريب أسرع: أقل أشجار + early stop أبكر (غيّر لـ False للدقة الأعلى)
FAST_TRAIN = False
LGBM_N_ESTIMATORS = 1200 if FAST_TRAIN else 4000
LGBM_EARLY_STOPPING = 50 if FAST_TRAIN else 100
LGBM_LEARNING_RATE = 0.08 if FAST_TRAIN else 0.05
LGBM_LOG_EVERY = 100 if FAST_TRAIN else 200

BASE_FEATURES_NUMERIC = [
    "distance_km_proxy",
    "temp_c",
    "rain_mm",
    "pickup_hour",
    "pickup_dow",
    "pickup_month",
    "pickup_dayofyear",
    "pickup_minute",
    "is_weekend",
    "is_rush_hour",
    "od_hour_median_duration",
    "pu_hour_slowdown_index",
    "distance_bucket_median_duration",
]

BASE_FEATURES_CATEGORICAL = [
    "PULocationID",
    "DOLocationID",
    "weather_code",
    "pickup_15min_bucket",
    "distance_bucket_label",
]

ALL_FEATURES = BASE_FEATURES_NUMERIC + BASE_FEATURES_CATEGORICAL
CAT_COLS = BASE_FEATURES_CATEGORICAL

C:\Users\A Store\AppData\Roaming\Python\Python312\site-packages\cupy\_environment.py:215: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


In [2]:
# ==========================
# Artifact schema
# ==========================

@dataclass
class EtaBaselineTables:
    global_median_duration: float
    global_slowdown_index: float
    od_hour_median: pd.DataFrame
    od_15min_median: pd.DataFrame
    distance_bucket_median: pd.DataFrame


@dataclass
class EtaModelArtifact:
    model_p50: Any
    model_p90: Any
    features: List[str]
    categorical_features: List[str]
    target: str
    model_version: str
    feature_version: str
    training_start: datetime
    training_end: datetime
    valid_start: datetime
    valid_end: datetime
    min_duration: float
    max_duration: float
    metrics_valid: Dict[str, float]
    metrics_test: Optional[Dict[str, float]]
    baselines: EtaBaselineTables
    congestion_stats: pd.DataFrame
    categorical_levels: Dict[str, List[str]]
    fillna_policy: Dict[str, Any]
    dtype_schema: Dict[str, str]
    clipping_rules: Dict[str, float]


def save_eta_artifact(artifact: EtaModelArtifact, path: str) -> None:
    joblib.dump(artifact, path)


def load_eta_artifact(path: str) -> EtaModelArtifact:
    return joblib.load(path)

In [3]:
# ==========================
# Data loading & time splits
# ==========================

def build_lazy_frame(clean_path: str) -> pl.LazyFrame:
    lf = (
        pl.scan_parquet(clean_path)
        .with_columns([
            (
                pl.col("tpep_dropoff_datetime").cast(pl.Datetime)
                - pl.col("tpep_pickup_datetime").cast(pl.Datetime)
            ).dt.total_seconds().alias("duration_sec"),
            pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),
            pl.col("tpep_pickup_datetime").dt.weekday().alias("pickup_dow"),
            pl.col("tpep_pickup_datetime").dt.month().alias("pickup_month"),
            pl.col("tpep_pickup_datetime").dt.ordinal_day().alias("pickup_dayofyear"),
            pl.col("tpep_pickup_datetime").dt.minute().alias("pickup_minute"),
            (pl.col("tpep_pickup_datetime").dt.weekday() >= 5).cast(pl.Int8).alias("is_weekend"),
            (
                (pl.col("tpep_pickup_datetime").dt.hour().is_between(7, 9, closed="both"))
                | (pl.col("tpep_pickup_datetime").dt.hour().is_between(16, 19, closed="both"))
            ).cast(pl.Int8).alias("is_rush_hour"),
        ])
        .filter(
            (pl.col("duration_sec") > MIN_DURATION)
            & (pl.col("duration_sec") < MAX_DURATION)
            & (pl.col("trip_distance") > 0.1)
            & (pl.col("trip_distance") < 50)
        )
        .with_columns([
            pl.col("trip_distance").alias("distance_km_proxy"),
            ((pl.col("pickup_hour") * 60 + pl.col("pickup_minute")) // 15)
            .cast(pl.Int16)
            .alias("pickup_15min_bucket"),
        ])
        .select([
            "tpep_pickup_datetime",
            "PULocationID",
            "DOLocationID",
            "distance_km_proxy",
            "temp_c",
            "rain_mm",
            "weather_code",
            "pickup_hour",
            "pickup_dow",
            "pickup_month",
            "pickup_dayofyear",
            "pickup_minute",
            "pickup_15min_bucket",
            "is_weekend",
            "is_rush_hour",
            "duration_sec",
        ])
    )
    return lf


def collect_time_slice(
    lf: pl.LazyFrame,
    start_dt: Optional[datetime],
    end_dt: Optional[datetime],
    max_rows: Optional[int] = None,
    random_sample: bool = True,
    seed: int = SEED,
) -> pd.DataFrame:
    part = lf
    if start_dt is not None:
        part = part.filter(pl.col("tpep_pickup_datetime") >= pl.lit(start_dt))
    if end_dt is not None:
        part = part.filter(pl.col("tpep_pickup_datetime") < pl.lit(end_dt))

    pdf = part.collect(engine="streaming").to_pandas()

    if max_rows is not None and len(pdf) > max_rows:
        if random_sample:
            pdf = pdf.sample(n=max_rows, random_state=seed)
        else:
            pdf = pdf.head(max_rows)

    int_cols = [
        "PULocationID",
        "DOLocationID",
        "weather_code",
        "pickup_hour",
        "pickup_dow",
        "pickup_month",
        "pickup_dayofyear",
        "pickup_minute",
        "pickup_15min_bucket",
        "is_weekend",
        "is_rush_hour",
    ]
    float_cols = ["distance_km_proxy", "temp_c", "rain_mm", "duration_sec"]

    for c in int_cols:
        if c in pdf.columns:
            pdf[c] = pd.to_numeric(pdf[c], downcast="integer")
    for c in float_cols:
        if c in pdf.columns:
            pdf[c] = pd.to_numeric(pdf[c], downcast="float")

    return pdf


def clean_eta_data(df: pd.DataFrame, report: bool = True) -> pd.DataFrame:
    """تنظيف البيانات ومعالجة الشواذ قبل النموذج."""
    n_before = len(df)
    out = df.copy()
    out["duration_sec"] = out["duration_sec"].clip(*DURATION_CLIP)
    out["distance_km_proxy"] = out["distance_km_proxy"].clip(*DISTANCE_KM_CLIP)
    if "temp_c" in out.columns:
        out["temp_c"] = pd.to_numeric(out["temp_c"], errors="coerce").fillna(0.0).clip(*TEMP_C_CLIP)
    if "rain_mm" in out.columns:
        out["rain_mm"] = pd.to_numeric(out["rain_mm"], errors="coerce").fillna(0.0).clip(*RAIN_MM_CLIP)
    sec_per_km = out["duration_sec"] / np.maximum(out["distance_km_proxy"], 0.01)
    mask_ok = (sec_per_km >= SPEED_SEC_PER_KM_CLIP[0]) & (sec_per_km <= SPEED_SEC_PER_KM_CLIP[1])
    out = out.loc[mask_ok].copy()
    if report and n_before > 0:
        print(f"  Clean: {n_before} -> {len(out)} rows (dropped {(1 - len(out)/n_before)*100:.2f}% outliers/speed)")
    return out


def load_train_valid_test(lf: pl.LazyFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df = collect_time_slice(lf, start_dt=None, end_dt=TRAIN_END, max_rows=MAX_TRAIN_ROWS, random_sample=True)
    valid_df = collect_time_slice(lf, start_dt=TRAIN_END, end_dt=VALID_END, max_rows=MAX_VALID_ROWS, random_sample=True)
    test_df = collect_time_slice(lf, start_dt=VALID_END, end_dt=None, max_rows=MAX_TEST_ROWS, random_sample=True)

    train_df = clean_eta_data(train_df, report=CLEAN_REPORT)
    valid_df = clean_eta_data(valid_df, report=CLEAN_REPORT)
    test_df = clean_eta_data(test_df, report=CLEAN_REPORT)

    print("Train time range:", train_df["tpep_pickup_datetime"].min(), "->", train_df["tpep_pickup_datetime"].max())
    print("Valid time range:", valid_df["tpep_pickup_datetime"].min(), "->", valid_df["tpep_pickup_datetime"].max())
    print("Test  time range:", test_df["tpep_pickup_datetime"].min(), "->", test_df["tpep_pickup_datetime"].max())

    return train_df, valid_df, test_df

In [7]:
# تنظيف البيانات ومعالجة الشواذ مُعرّف في الخلية السابقة (clean_eta_data)
# ويُستدعى تلقائياً داخل load_train_valid_test

In [4]:
# ==========================
# Feature engineering & baselines
# ==========================

def add_distance_bucket(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    bins = DIST_BUCKET_EDGES
    labels = [f"{bins[i]}-{bins[i+1]}km" for i in range(len(bins) - 1)]
    df["distance_bucket_label"] = pd.cut(
        df["distance_km_proxy"],
        bins=bins + [np.inf],
        labels=labels + [f">{bins[-1]}km"],
        right=False,
    ).astype("category")
    return df


def compute_baseline_tables(train_df: pd.DataFrame) -> EtaBaselineTables:
    df = train_df.copy()
    global_median = float(df["duration_sec"].median())

    slowdown = df["duration_sec"] / np.maximum(df["distance_km_proxy"], 0.1)
    global_slowdown_index = float(slowdown.median())

    od_hour = (
        df.groupby(["PULocationID", "DOLocationID", "pickup_hour"])["duration_sec"]
        .agg(median_duration_sec=("median"), count=("size"))
        .reset_index()
    )

    od_15 = (
        df.groupby(["PULocationID", "DOLocationID", "pickup_15min_bucket"])["duration_sec"]
        .agg(median_duration_sec=("median"), count=("size"))
        .reset_index()
    )

    df = add_distance_bucket(df)
    dist_med = (
        df.groupby("distance_bucket_label")["duration_sec"]
        .agg(median_duration_sec=("median"), count=("size"))
        .reset_index()
    )

    return EtaBaselineTables(
        global_median_duration=global_median,
        global_slowdown_index=global_slowdown_index,
        od_hour_median=od_hour,
        od_15min_median=od_15,
        distance_bucket_median=dist_med,
    )


def build_congestion_proxy(train_df: pd.DataFrame) -> pd.DataFrame:
    df = train_df.copy()
    df["duration_per_km"] = df["duration_sec"] / np.maximum(df["distance_km_proxy"], 0.1)
    cong = (
        df.groupby(["PULocationID", "pickup_hour"])["duration_per_km"]
        .median()
        .reset_index()
        .rename(columns={"duration_per_km": "median_duration_per_km"})
    )
    return cong


def merge_feature_stats(
    df: pd.DataFrame,
    baselines: EtaBaselineTables,
    cong_stats: pd.DataFrame,
) -> pd.DataFrame:
    df = df.copy()

    df = add_distance_bucket(df)
    df = df.merge(
        baselines.distance_bucket_median,
        on="distance_bucket_label",
        how="left",
        suffixes=("", "_dist_bucket"),
    )
    df.rename(columns={"median_duration_sec": "distance_bucket_median_duration"}, inplace=True)

    df = df.merge(
        baselines.od_hour_median,
        on=["PULocationID", "DOLocationID", "pickup_hour"],
        how="left",
    )
    df.rename(columns={"median_duration_sec": "od_hour_median_duration"}, inplace=True)

    df = df.merge(
        cong_stats,
        on=["PULocationID", "pickup_hour"],
        how="left",
    )
    df.rename(columns={"median_duration_per_km": "pu_hour_slowdown_index"}, inplace=True)

    if "distance_bucket_median_duration" in df.columns:
        df["distance_bucket_median_duration"] = df["distance_bucket_median_duration"].fillna(
            baselines.global_median_duration
        )

    if "od_hour_median_duration" in df.columns:
        df["od_hour_median_duration"] = df["od_hour_median_duration"].fillna(
            df.get("distance_bucket_median_duration", baselines.global_median_duration)
        )
        df["od_hour_median_duration"] = df["od_hour_median_duration"].fillna(baselines.global_median_duration)

    if "pu_hour_slowdown_index" in df.columns:
        df["pu_hour_slowdown_index"] = df["pu_hour_slowdown_index"].fillna(baselines.global_slowdown_index)

    return df

In [5]:
def default_fillna_policy() -> Dict[str, Any]:
    return {
        "numeric": 0.0,
        "categorical_missing": CATEGORICAL_MISSING_TOKEN,
        "categorical_unknown": CATEGORICAL_UNK_TOKEN,
    }


def build_categorical_levels(df: pd.DataFrame, categorical_cols: List[str]) -> Dict[str, List[str]]:
    levels: Dict[str, List[str]] = {}
    for col in categorical_cols:
        s = df[col].copy()
        s = s.astype("string").fillna(CATEGORICAL_MISSING_TOKEN)
        uniq = sorted(pd.Series(s).dropna().unique().tolist())

        if CATEGORICAL_MISSING_TOKEN not in uniq:
            uniq.append(CATEGORICAL_MISSING_TOKEN)
        if CATEGORICAL_UNK_TOKEN not in uniq:
            uniq.append(CATEGORICAL_UNK_TOKEN)

        levels[col] = uniq
    return levels


def apply_categorical_schema(
    df: pd.DataFrame,
    categorical_cols: List[str],
    categorical_levels: Optional[Dict[str, List[str]]] = None,
    missing_token: str = CATEGORICAL_MISSING_TOKEN,
    unk_token: str = CATEGORICAL_UNK_TOKEN,
) -> pd.DataFrame:
    df = df.copy()

    for col in categorical_cols:
        if col not in df.columns:
            df[col] = missing_token

        s = df[col].astype("string").fillna(missing_token)

        if categorical_levels is None:
            levels = sorted(pd.Series(s).dropna().unique().tolist())
            if missing_token not in levels:
                levels.append(missing_token)
            if unk_token not in levels:
                levels.append(unk_token)
        else:
            levels = list(categorical_levels[col])
            known = set(levels)
            s = s.where(s.isin(known), unk_token)

        df[col] = pd.Categorical(s, categories=levels)

    return df


def apply_fillna(
    df: pd.DataFrame,
    policy: Dict[str, Any],
    categorical_cols: Optional[List[str]] = None,
    categorical_levels: Optional[Dict[str, List[str]]] = None,
) -> pd.DataFrame:
    df = df.copy()
    categorical_cols = categorical_cols or []

    numeric_cols = [c for c in df.columns if c not in categorical_cols and pd.api.types.is_numeric_dtype(df[c])]
    df[numeric_cols] = df[numeric_cols].fillna(policy["numeric"])

    df = apply_categorical_schema(
        df=df,
        categorical_cols=categorical_cols,
        categorical_levels=categorical_levels,
        missing_token=policy["categorical_missing"],
        unk_token=policy["categorical_unknown"],
    )

    return df


def build_training_features(
    df: pd.DataFrame,
    baselines: EtaBaselineTables,
    cong_stats: pd.DataFrame,
    fillna_policy: Dict[str, Any],
    categorical_levels: Optional[Dict[str, List[str]]] = None,
) -> Tuple[pd.DataFrame, np.ndarray]:
    df = merge_feature_stats(df, baselines, cong_stats)
    df = apply_fillna(
        df,
        fillna_policy,
        categorical_cols=CAT_COLS,
        categorical_levels=categorical_levels,
    )

    X = df[ALL_FEATURES].copy()
    y = df[TARGET].clip(MIN_DURATION, MAX_DURATION).values.astype("float32")
    return X, y

In [6]:
# ==========================
# Quantile model training
# ==========================

def train_quantile_model(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_valid: pd.DataFrame,
    y_valid: np.ndarray,
    alpha: float,
    categorical_features: List[str],
    seed: int = SEED,
) -> lgb.LGBMRegressor:
    model = lgb.LGBMRegressor(
        objective="quantile",
        alpha=alpha,
        n_estimators=LGBM_N_ESTIMATORS,
        learning_rate=LGBM_LEARNING_RATE,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=100,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="l1",
        categorical_feature=categorical_features,
        callbacks=[
            lgb.early_stopping(LGBM_EARLY_STOPPING),
            lgb.log_evaluation(LGBM_LOG_EVERY),
        ],
    )
    print(f"[alpha={alpha}] Best iteration:", model.best_iteration_)
    return model


def train_eta_models(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
) -> Tuple[lgb.LGBMRegressor, lgb.LGBMRegressor, EtaBaselineTables, Dict[str, Any]]:
    baselines = compute_baseline_tables(train_df)
    cong_stats = build_congestion_proxy(train_df)
    fillna_policy = default_fillna_policy()

    X_train, y_train = build_training_features(
        train_df,
        baselines,
        cong_stats,
        fillna_policy,
        categorical_levels=None,
    )

    categorical_levels = build_categorical_levels(X_train, CAT_COLS)

    X_train, y_train = build_training_features(
        train_df,
        baselines,
        cong_stats,
        fillna_policy,
        categorical_levels=categorical_levels,
    )
    X_valid, y_valid = build_training_features(
        valid_df,
        baselines,
        cong_stats,
        fillna_policy,
        categorical_levels=categorical_levels,
    )

    model_p50 = train_quantile_model(
        X_train, y_train, X_valid, y_valid,
        alpha=0.5,
        categorical_features=CAT_COLS,
    )
    model_p90 = train_quantile_model(
        X_train, y_train, X_valid, y_valid,
        alpha=0.9,
        categorical_features=CAT_COLS,
    )

    dtype_schema = {col: str(dtype) for col, dtype in X_train.dtypes.items()}

    return model_p50, model_p90, baselines, {
        "fillna_policy": fillna_policy,
        "dtype_schema": dtype_schema,
        "categorical_levels": categorical_levels,
        "congestion_stats": cong_stats,
    }

In [7]:
# ==========================
# Evaluation helpers
# ==========================

def compute_global_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = y_true.astype("float32")
    y_pred = y_pred.astype("float32")

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100.0)

    acc_1m = float((np.abs(y_true - y_pred) <= 60).mean() * 100.0)
    acc_2m = float((np.abs(y_true - y_pred) <= 120).mean() * 100.0)
    acc_5m = float((np.abs(y_true - y_pred) <= 300).mean() * 100.0)

    return {
        "MAE_sec": mae,
        "RMSE_sec": rmse,
        "MAPE_pct": mape,
        "ACC_1min_pct": acc_1m,
        "ACC_2min_pct": acc_2m,
        "ACC_5min_pct": acc_5m,
    }


def evaluate_segmented(
    df: pd.DataFrame,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    group_cols: List[str],
) -> pd.DataFrame:
    eval_df = df.copy()
    eval_df["y_true"] = y_true
    eval_df["y_pred"] = y_pred
    eval_df["abs_err"] = np.abs(eval_df["y_true"] - eval_df["y_pred"])
    eval_df["ape"] = np.abs((eval_df["y_true"] - eval_df["y_pred"]) / np.maximum(eval_df["y_true"], 1.0)) * 100.0

    grouped = (
        eval_df.groupby(group_cols)
        .agg(
            count=("y_true", "size"),
            mae_sec=("abs_err", "mean"),
            mape_pct=("ape", "mean"),
        )
        .reset_index()
        .sort_values("mae_sec", ascending=False)
    )
    return grouped


def build_lightweight_fallback_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    dt_col = None
    for c in ["tpep_pickup_datetime", "pickup_datetime"]:
        if c in out.columns:
            dt_col = c
            break

    if dt_col is not None:
        dt = pd.to_datetime(out[dt_col], errors="coerce")
        valid_dt = ~dt.isna()

        if valid_dt.any():
            out.loc[valid_dt, "pickup_hour"] = dt[valid_dt].dt.hour.astype("int16")
            out.loc[valid_dt, "pickup_minute"] = dt[valid_dt].dt.minute.astype("int16")
            out.loc[valid_dt, "pickup_15min_bucket"] = (
                (out.loc[valid_dt, "pickup_hour"] * 60 + out.loc[valid_dt, "pickup_minute"]) // 15
            ).astype("int16")

    if "distance_km_proxy" in out.columns:
        out["distance_km_proxy"] = pd.to_numeric(out["distance_km_proxy"], errors="coerce").clip(0.1, 50.0)
    elif "expected_distance_km" in out.columns:
        out["distance_km_proxy"] = pd.to_numeric(out["expected_distance_km"], errors="coerce").clip(0.1, 50.0)

    if "distance_km_proxy" in out.columns:
        out = add_distance_bucket(out)

    return out


def baseline_predict(df: pd.DataFrame, baselines: EtaBaselineTables) -> Dict[str, np.ndarray]:
    tmp_df = build_lightweight_fallback_features(df)

    n = len(tmp_df)
    pred = np.full(n, baselines.global_median_duration, dtype="float32")
    source = np.array(["global_median"] * n, dtype=object)

    if "distance_bucket_label" in tmp_df.columns:
        dist_tmp = tmp_df.merge(
            baselines.distance_bucket_median,
            on="distance_bucket_label",
            how="left",
        )

        dist_ok = dist_tmp["median_duration_sec"].notna()
        if "count" in dist_tmp.columns:
            dist_ok &= dist_tmp["count"].fillna(0) >= MIN_SUPPORT_DIST_BUCKET

        pred[dist_ok.values] = dist_tmp.loc[dist_ok, "median_duration_sec"].astype("float32").values
        source[dist_ok.values] = "distance_bucket"

    if all(c in tmp_df.columns for c in ["PULocationID", "DOLocationID", "pickup_hour"]):
        odh_tmp = tmp_df.merge(
            baselines.od_hour_median,
            on=["PULocationID", "DOLocationID", "pickup_hour"],
            how="left",
        )

        odh_ok = odh_tmp["median_duration_sec"].notna()
        if "count" in odh_tmp.columns:
            odh_ok &= odh_tmp["count"].fillna(0) >= MIN_SUPPORT_OD_HOUR

        pred[odh_ok.values] = odh_tmp.loc[odh_ok, "median_duration_sec"].astype("float32").values
        source[odh_ok.values] = "od_hour"

    if all(c in tmp_df.columns for c in ["PULocationID", "DOLocationID", "pickup_15min_bucket"]):
        od15_tmp = tmp_df.merge(
            baselines.od_15min_median,
            on=["PULocationID", "DOLocationID", "pickup_15min_bucket"],
            how="left",
        )

        od15_ok = od15_tmp["median_duration_sec"].notna()
        if "count" in od15_tmp.columns:
            od15_ok &= od15_tmp["count"].fillna(0) >= MIN_SUPPORT_OD_15MIN

        pred[od15_ok.values] = od15_tmp.loc[od15_ok, "median_duration_sec"].astype("float32").values
        source[od15_ok.values] = "od_15min"

    return {
        "hierarchical_baseline_pred": pred.astype("float32"),
        "fallback_source": source,
        "global_median_pred": np.full(n, baselines.global_median_duration, dtype="float32"),
    }

In [8]:
# ==========================
# End-to-end train + save
# ==========================

def train_and_save_eta_pipeline(train_df=None, valid_df=None, test_df=None):
    if train_df is None or valid_df is None or test_df is None:
        print("[1/5] Loading data from", CLEAN_PATH)
        lf = build_lazy_frame(CLEAN_PATH)
        train_df, valid_df, test_df = load_train_valid_test(lf)
    else:
        print("[1/5] Using provided train/valid/test data.")

    print("[2/5] Data shapes after clean:", train_df.shape, valid_df.shape, test_df.shape)
    if len(train_df) < 1000:
        raise ValueError(f"Train data too small ({len(train_df)} rows). Check CLEAN_PATH and that parquet exists.")
    assert train_df[TARGET].notna().all(), "قيم duration_sec ناقصة"
    print("  ✓ الخطوة 1: تحميل البيانات نجح")

    print("[3/5] Training P50 + P90 models...")
    model_p50, model_p90, baselines, extra = train_eta_models(train_df, valid_df)
    assert model_p50 is not None and model_p90 is not None and model_p50.best_iteration_ is not None
    print("  ✓ الخطوة 2: التدريب نجح")

    fillna_policy = extra["fillna_policy"]
    dtype_schema = extra["dtype_schema"]
    categorical_levels = extra["categorical_levels"]
    cong_stats = extra["congestion_stats"]

    X_valid, y_valid = build_training_features(
        valid_df, baselines, cong_stats, fillna_policy, categorical_levels
    )
    X_test, y_test = build_training_features(
        test_df, baselines, cong_stats, fillna_policy, categorical_levels
    )

    pred_p50_valid = np.clip(
        model_p50.predict(X_valid, num_iteration=model_p50.best_iteration_),
        MIN_DURATION,
        MAX_DURATION,
    )
    pred_p50_test = np.clip(
        model_p50.predict(X_test, num_iteration=model_p50.best_iteration_),
        MIN_DURATION,
        MAX_DURATION,
    )

    metrics_valid = compute_global_metrics(y_valid, pred_p50_valid)
    metrics_test = compute_global_metrics(y_test, pred_p50_test)

    print("[4/5] Evaluation:")
    print("VALID metrics:", metrics_valid)
    print("TEST  metrics:", metrics_test)
    assert "MAE_sec" in metrics_valid and "MAE_sec" in metrics_test
    print("  ✓ الخطوة 3: التقييم نجح")

    baseline_preds = baseline_predict(test_df, baselines)
    baseline_metrics = {
        "hierarchical_baseline_pred": compute_global_metrics(y_test, baseline_preds["hierarchical_baseline_pred"]),
        "global_median_pred": compute_global_metrics(y_test, baseline_preds["global_median_pred"]),
    }
    print("Baseline metrics on TEST:", baseline_metrics)

    fi = pd.DataFrame({
        "feature": ALL_FEATURES,
        "importance": model_p50.feature_importances_,
    }).sort_values("importance", ascending=False)
    fi.to_csv(FI_PATH, index=False)
    print("Feature importance saved to:", FI_PATH)

    artifact = EtaModelArtifact(
        model_p50=model_p50,
        model_p90=model_p90,
        features=ALL_FEATURES,
        categorical_features=CAT_COLS,
        target=TARGET,
        model_version=MODEL_VERSION,
        feature_version=FEATURE_VERSION,
        training_start=train_df["tpep_pickup_datetime"].min(),
        training_end=train_df["tpep_pickup_datetime"].max(),
        valid_start=valid_df["tpep_pickup_datetime"].min(),
        valid_end=valid_df["tpep_pickup_datetime"].max(),
        min_duration=MIN_DURATION,
        max_duration=MAX_DURATION,
        metrics_valid=metrics_valid,
        metrics_test=metrics_test,
        baselines=baselines,
        congestion_stats=cong_stats,
        categorical_levels=categorical_levels,
        fillna_policy=fillna_policy,
        dtype_schema=dtype_schema,
        clipping_rules={"min": MIN_DURATION, "max": MAX_DURATION},
    )

    save_eta_artifact(artifact, MODEL_PATH)
    assert os.path.isfile(MODEL_PATH), "ملف النموذج لم يُحفظ"
    print("[5/5] ETA artifact saved to:", MODEL_PATH)
    print("  ✓ الخطوة 4: الحفظ نجح")

In [9]:
# ==========================
# Rolling backtest helper
# ==========================

def rolling_backtest(
    lf: pl.LazyFrame,
    train_end_dates: List[datetime],
    valid_window_days: int = 14,
    max_train_rows: int = 2_000_000,
    max_valid_rows: int = 300_000,
) -> pd.DataFrame:
    results: List[Dict[str, Any]] = []

    for te in train_end_dates:
        ve = te + timedelta(days=valid_window_days)
        print(f"Rolling window: train < {te}, valid [{te}, {ve})")

        train_df = collect_time_slice(lf, start_dt=None, end_dt=te, max_rows=max_train_rows, random_sample=True)
        valid_df = collect_time_slice(lf, start_dt=te, end_dt=ve, max_rows=max_valid_rows, random_sample=True)

        model_p50, model_p90, baselines, extra = train_eta_models(train_df, valid_df)
        fillna_policy = extra["fillna_policy"]
        cong_stats = extra["congestion_stats"]
        categorical_levels = extra["categorical_levels"]

        X_valid, y_valid = build_training_features(
            valid_df, baselines, cong_stats, fillna_policy, categorical_levels
        )
        pred_valid = np.clip(
            model_p50.predict(X_valid, num_iteration=model_p50.best_iteration_),
            MIN_DURATION,
            MAX_DURATION,
        )
        metrics_valid = compute_global_metrics(y_valid, pred_valid)

        results.append({
            "train_end": te,
            "valid_start": te,
            "valid_end": ve,
            **metrics_valid,
        })

        del train_df, valid_df, model_p50, model_p90, baselines, extra, X_valid, y_valid, pred_valid
        gc.collect()

    return pd.DataFrame(results)

In [10]:
# ==========================
# Inference: feature builder
# ==========================

def build_inference_features(raw_df: pd.DataFrame, artifact: EtaModelArtifact) -> Tuple[pd.DataFrame, List[str]]:
    df = raw_df.copy()

    dt_col = None
    for c in ["tpep_pickup_datetime", "pickup_datetime"]:
        if c in df.columns:
            dt_col = c
            break
    if dt_col is None:
        raise ValueError("raw_df must contain a pickup datetime column (e.g. 'tpep_pickup_datetime').")

    dt = pd.to_datetime(df[dt_col], errors="coerce")
    if dt.isna().any():
        raise ValueError("pickup datetime contains invalid/unparseable values.")

    df["pickup_hour"] = dt.dt.hour.astype("int16")
    df["pickup_dow"] = dt.dt.weekday.astype("int8")
    df["pickup_month"] = dt.dt.month.astype("int8")
    df["pickup_dayofyear"] = dt.dt.dayofyear.astype("int16")
    df["pickup_minute"] = dt.dt.minute.astype("int8")
    df["is_weekend"] = (df["pickup_dow"] >= 5).astype("int8")
    df["is_rush_hour"] = (
        ((df["pickup_hour"] >= 7) & (df["pickup_hour"] <= 9))
        | ((df["pickup_hour"] >= 16) & (df["pickup_hour"] <= 19))
    ).astype("int8")
    df["pickup_15min_bucket"] = ((df["pickup_hour"] * 60 + df["pickup_minute"]) // 15).astype("int16")

    if "expected_distance_km" not in df.columns:
        raise ValueError("raw_df must contain 'expected_distance_km' from routing engine.")

    df["distance_km_proxy"] = pd.to_numeric(df["expected_distance_km"], errors="coerce").clip(0.1, 50.0)

    cong_stats = artifact.congestion_stats
    df = merge_feature_stats(df, artifact.baselines, cong_stats)
    df = apply_fillna(
        df,
        artifact.fillna_policy,
        categorical_cols=artifact.categorical_features,
        categorical_levels=artifact.categorical_levels,
    )

    X = df[artifact.features].copy()
    return X, artifact.features


def _clip_and_fix_quantiles(
    p50: np.ndarray,
    p90: np.ndarray,
    min_v: float,
    max_v: float,
) -> Tuple[np.ndarray, np.ndarray]:
    p50 = np.clip(p50, min_v, max_v)
    p90 = np.clip(p90, min_v, max_v)
    p90 = np.maximum(p90, p50 + 1.0)
    return p50, p90

In [11]:
def predict_eta(
    artifact_path: str,
    raw_input: pd.DataFrame,
    allow_baseline_fallback: bool = True,
) -> pd.DataFrame:
    artifact: EtaModelArtifact = load_eta_artifact(artifact_path)
    df = raw_input.copy()

    if len(df) == 0:
        return pd.DataFrame(columns=[
            "eta_seconds_p50",
            "eta_seconds_p90",
            "eta_confidence",
            "fallback_used",
            "fallback_source",
            "model_version",
            "feature_status",
            "warnings",
        ])

    required_cols = ["PULocationID", "DOLocationID"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if "temp_c" not in df.columns:
        df["temp_c"] = 0.0
    if "rain_mm" not in df.columns:
        df["rain_mm"] = 0.0
    if "weather_code" not in df.columns:
        df["weather_code"] = -1

    has_distance = "expected_distance_km" in df.columns
    invalid_distance = False
    if has_distance:
        dist = pd.to_numeric(df["expected_distance_km"], errors="coerce")
        invalid_mask = (dist <= 0) | ~np.isfinite(dist)
        if invalid_mask.any():
            invalid_distance = True
            dist[invalid_mask] = 0.5
        df["expected_distance_km"] = dist

    feature_status = "ok"
    warnings_list: List[str] = []
    if invalid_distance:
        warnings_list.append("expected_distance_km_invalid_rows_replaced_with_0.5km")

    try:
        X, _ = build_inference_features(df, artifact)
        model_failed = False
    except Exception as e:
        feature_status = "incomplete"
        warnings_list.append(f"feature_builder_error: {e}")
        model_failed = True
        X = None

    n = len(df)
    eta_p50 = np.full(n, np.nan, dtype="float32")
    eta_p90 = np.full(n, np.nan, dtype="float32")
    fallback_used = np.zeros(n, dtype=bool)
    fallback_source = np.array([""] * n, dtype=object)

    if not model_failed:
        try:
            p50_raw = artifact.model_p50.predict(X, num_iteration=artifact.model_p50.best_iteration_)
            p90_raw = artifact.model_p90.predict(X, num_iteration=artifact.model_p90.best_iteration_)
            p50_raw, p90_raw = _clip_and_fix_quantiles(
                p50_raw, p90_raw, artifact.min_duration, artifact.max_duration
            )
            eta_p50[:] = p50_raw
            eta_p90[:] = p90_raw
        except Exception as e:
            feature_status = "model_inference_error"
            warnings_list.append(f"model_inference_error: {e}")
            model_failed = True

    if model_failed or np.any(np.isnan(eta_p50)):
        if allow_baseline_fallback:
            warnings_list.append("using_baseline_fallback")

            base_preds = baseline_predict(df, artifact.baselines)
            base_best = base_preds["hierarchical_baseline_pred"]
            base_source = base_preds["fallback_source"]

            mask = np.isnan(eta_p50)
            eta_p50[mask] = base_best[mask]
            eta_p90[mask] = eta_p50[mask] * 1.3
            fallback_used[mask] = True
            fallback_source[mask] = base_source[mask]
        else:
            warnings_list.append("no_fallback_allowed_and_model_failed")

    width = eta_p90 - eta_p50
    ratio = width / np.maximum(eta_p50, 1.0)
    eta_confidence = 1.0 - np.clip(ratio, 0.0, 1.0)

    eta_confidence = np.clip(eta_confidence * np.where(fallback_used, 0.7, 1.0), 0.0, 1.0)

    out = pd.DataFrame({
        "eta_seconds_p50": eta_p50.astype("float32"),
        "eta_seconds_p90": eta_p90.astype("float32"),
        "eta_confidence": eta_confidence.astype("float32"),
        "fallback_used": fallback_used,
        "fallback_source": fallback_source,
        "model_version": artifact.model_version,
        "feature_status": feature_status,
        "warnings": [list(warnings_list) for _ in range(n)],
    })

    return out

In [12]:
# (1) قراءة البيانات مرة واحدة
print("=" * 50)
print("(1) قراءة البيانات من", CLEAN_PATH)
lf = build_lazy_frame(CLEAN_PATH)
train_df, valid_df, test_df = load_train_valid_test(lf)
assert len(train_df) >= 1000, f"بيانات التدريب قليلة: {len(train_df)} صف. تأكد من المسار والملفات."
print("تم التحميل. الأحجام: train", train_df.shape, "| valid", valid_df.shape, "| test", test_df.shape)
print("  ✓ قراءة البيانات نجحت")
print("=" * 50)

# (2) بعدين كل الخطوات: تدريب + تقييم + حفظ
print("(2) تدريب النموذج + التقييم + حفظ الـ artifact...")
train_and_save_eta_pipeline(train_df=train_df, valid_df=valid_df, test_df=test_df)
print("=" * 50)
print("  ✓ كل الخطوات نجحت. النموذج محفوظ في:", MODEL_PATH)

(1) قراءة البيانات من C:\grad2_out\taxi_with_weather_FULL_dataset
  Clean: 3000000 -> 2529346 rows (dropped 15.69% outliers/speed)
  Clean: 500000 -> 400067 rows (dropped 19.99% outliers/speed)
  Clean: 500000 -> 422095 rows (dropped 15.58% outliers/speed)
Train time range: 2024-01-01 00:00:11 -> 2024-10-31 23:59:59
Valid time range: 2024-11-01 00:00:01 -> 2024-11-30 23:59:55
Test  time range: 2024-12-01 00:00:14 -> 2025-11-30 23:51:19
تم التحميل. الأحجام: train (2529346, 16) | valid (400067, 16) | test (422095, 16)
  ✓ قراءة البيانات نجحت
(2) تدريب النموذج + التقييم + حفظ الـ artifact...
[1/5] Using provided train/valid/test data.
[2/5] Data shapes after clean: (2529346, 16) (400067, 16) (422095, 16)
  ✓ الخطوة 1: تحميل البيانات نجح
[3/5] Training P50 + P90 models...


C:\Users\A Store\AppData\Local\Temp\ipykernel_16972\1986774785.py:39: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("distance_bucket_label")["duration_sec"]


Training until validation scores don't improve for 100 rounds
[200]	valid_0's l1: 190.554	valid_0's quantile: 95.2768
[400]	valid_0's l1: 186.169	valid_0's quantile: 93.0843
[600]	valid_0's l1: 184.696	valid_0's quantile: 92.3479
[800]	valid_0's l1: 184.133	valid_0's quantile: 92.0666
[1000]	valid_0's l1: 183.657	valid_0's quantile: 91.8284
[1200]	valid_0's l1: 183.341	valid_0's quantile: 91.6704
[1400]	valid_0's l1: 183.129	valid_0's quantile: 91.5646
[1600]	valid_0's l1: 182.949	valid_0's quantile: 91.4745
[1800]	valid_0's l1: 182.85	valid_0's quantile: 91.4249
Early stopping, best iteration is:
[1768]	valid_0's l1: 182.806	valid_0's quantile: 91.4029
[alpha=0.5] Best iteration: 1768
Training until validation scores don't improve for 100 rounds
[200]	valid_0's l1: 308.711	valid_0's quantile: 50.976
[400]	valid_0's l1: 302.526	valid_0's quantile: 50.4848
Early stopping, best iteration is:
[477]	valid_0's l1: 301.095	valid_0's quantile: 50.4395
[alpha=0.9] Best iteration: 477
  ✓ الخطو

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\A Store\AppData\Local\Temp\ipykernel_16972\324498986.py:66: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[20 18 13 ... 23 12  6]' has dtype incompatible with int8, please explicitly cast to a compatible dtype first.
  out.loc[valid_dt, "pickup_hour"] = dt[valid_dt].dt.hour.astype("int16")
C:\Users\A Store\AppData\Local\Temp\ipykernel_16972\324498986.py:67: FutureWarning: Setting

[4/5] Evaluation:
VALID metrics: {'MAE_sec': 182.80536, 'RMSE_sec': 330.73322, 'MAPE_pct': 18.35608184337616, 'ACC_1min_pct': 32.85849620188618, 'ACC_2min_pct': 56.48403892348032, 'ACC_5min_pct': 84.35836997302951}
TEST  metrics: {'MAE_sec': 185.20851, 'RMSE_sec': 338.15985, 'MAPE_pct': 17.95377880334854, 'ACC_1min_pct': 33.22237884836352, 'ACC_2min_pct': 56.61190016465487, 'ACC_5min_pct': 84.04292872457621}
  ✓ الخطوة 3: التقييم نجح


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Baseline metrics on TEST: {'hierarchical_baseline_pred': {'MAE_sec': 260.0004, 'RMSE_sec': 423.35934, 'MAPE_pct': 29.4836163520813, 'ACC_1min_pct': 20.821379073431338, 'ACC_2min_pct': 39.358912093249145, 'ACC_5min_pct': 73.38300619528779}, 'global_median_pred': {'MAE_sec': 553.09924, 'RMSE_sec': 875.64703, 'MAPE_pct': 67.99890398979187, 'ACC_1min_pct': 8.722443999573555, 'ACC_2min_pct': 17.24682832063872, 'ACC_5min_pct': 42.47100771153413}}
Feature importance saved to: C:\grad2_out\lgbm_eta_feature_importance_v2.csv
[5/5] ETA artifact saved to: C:\grad2_out\eta_model_artifact.joblib
  ✓ الخطوة 4: الحفظ نجح
  ✓ كل الخطوات نجحت. النموذج محفوظ في: C:\grad2_out\eta_model_artifact.joblib


In [14]:
raw_df = pd.DataFrame({
    "pickup_datetime": ["2025-01-15 08:30:00"],
    "PULocationID": [15],
    "DOLocationID": [20],
    "expected_distance_km": [8.5],
    "temp_c": [22.0],
    "rain_mm": [0.0],
    "weather_code": [1],
})

pred = predict_eta(MODEL_PATH, raw_df)
print(pred)

   eta_seconds_p50  eta_seconds_p90  eta_confidence  fallback_used  \
0      1543.092529      1802.473145        0.831909          False   

  fallback_source      model_version feature_status warnings  
0                  eta_v1.0_quantile             ok       []  


In [17]:
raw_df = pd.DataFrame({
    "pickup_datetime": ["2025-01-16 14:20:00"],
    "PULocationID": [18],
    "DOLocationID": [21],
    "expected_distance_km": [17.8],
    "temp_c": [11.0],
    "rain_mm": [0.0],
    "weather_code": [2],
})

pred = predict_eta(MODEL_PATH, raw_df)
print(pred)

   eta_seconds_p50  eta_seconds_p90  eta_confidence  fallback_used  \
0      2471.633545      2644.366455        0.930114          False   

  fallback_source      model_version feature_status warnings  
0                  eta_v1.0_quantile             ok       []  
